# How to use the Matcher framework?

The `Matcher` framework provides a high-level interface for computing correspondences between shapes. It encapsulates the full functional map pipeline into a single, configurable class.

In [1]:
import gsops.backend as gs

from geomfum.dataset import NotebooksDataset
from geomfum.matcher import (
    FeatureMatcher,
    FeatureMatcherConfig,
    FunctionalMapMatcher,
    MatcherConfig,
    PreciseMatcher,
    QuickMatcher,
)
from geomfum.shape import TriangleMesh

[Load meshes](00_load_mesh_from_file.ipynb).

In [2]:
dataset = NotebooksDataset()

mesh_a = TriangleMesh.from_file(dataset.get_filename("cat-00"))
mesh_b = TriangleMesh.from_file(dataset.get_filename("lion-00"))

mesh_a.n_vertices, mesh_b.n_vertices

INFO:root:Data has already been downloaded... using cached file ('C:\Users\giuli\.geomfum\data\cat-00.off').
INFO:root:Data has already been downloaded... using cached file ('C:\Users\giuli\.geomfum\data\lion-00.off').


(7207, 5000)

## Feature Matcher

The simplest way to match shapes is computing features and performing nearest neighbor search, this routine is made by the Feature matcher.



In [3]:
# Basic usage with defaults
matcher = FeatureMatcher()
result = matcher(mesh_a, mesh_b)
p2p = result.p2p

print(f"P2P shape: {p2p.shape}")

P2P shape: (5000,)


The result contains:
- `p2p`: point-to-point correspondence
- `descr_a`, `descr_b`: computed descriptors
- `fmap`: functional map matrix (None for FeatureMatcher)
- `refined_fmap`: refined functional map (None for FeatureMatcher)

In [4]:
result.descr_a.shape, result.descr_b.shape

((400, 7207), (400, 5000))

## Functional Map Matcher

For more robust matching, use `FunctionalMapMatcher` which optimizes a functional map.

In [5]:
matcher = FunctionalMapMatcher()
result = matcher(mesh_a, mesh_b)

print(f"P2P shape: {result.p2p.shape}")
print(f"Fmap shape: {result.fmap.shape}")
print(f"Refined Fmap shape: {result.refined_fmap.shape}")

P2P shape: (5000,)
Fmap shape: (30, 30)
Refined Fmap shape: (60, 60)


## Using landmarks

[Set landmarks](./06_landmarks.ipynb) on both shapes for better matching.

In [6]:
mesh_a.set_landmarks(gs.array([2840, 1594, 5596, 6809, 3924, 7169]))
mesh_b.set_landmarks(gs.array([1334, 834, 4136, 4582, 3666, 4955]))

Use landmarks by adding `LandmarkWaveKernelSignature` to the descriptors list.

In [ ]:
from geomfum.descriptor.pipeline import ArangeSubsampler
from geomfum.descriptor.spectral import LandmarkWaveKernelSignature, WaveKernelSignature

config = MatcherConfig(
    descriptors=[
        WaveKernelSignature.from_registry(n_domain=200),
        LandmarkWaveKernelSignature.from_registry(n_domain=200),
    ],
    subsamplers=[ArangeSubsampler(subsample_step=5)],
)

matcher = FunctionalMapMatcher(config=config)
result = matcher(mesh_a, mesh_b)

result.p2p.shape

(5000,)

## Preset matchers

Several preset matchers are available for common use cases.

### QuickMatcher

Fast matching with reduced settings (smaller spectrum, fewer refinement iterations).

In [8]:
quick_matcher = QuickMatcher()

result = quick_matcher(mesh_a, mesh_b)

result.p2p.shape

(5000,)

### PreciseMatcher

High-quality matching with larger settings.

In [9]:
precise_matcher = PreciseMatcher(use_landmarks=True)

result = precise_matcher(mesh_a, mesh_b)

result.p2p.shape

(5000,)

## Custom configuration

Use `MatcherConfig` to fully customize the matching pipeline.

In [ ]:
from geomfum.descriptor.spectral import HeatKernelSignature
from geomfum.refine import IcpRefiner, ZoomOut

config = MatcherConfig(
    spectrum_size=100,  # Number of eigenfunctions to compute
    fmap_size=20,  # Size of functional map matrix
    descriptors=[  # Custom descriptors
        HeatKernelSignature.from_registry(n_domain=100),
        WaveKernelSignature.from_registry(n_domain=200),
    ],
    subsamplers=[ArangeSubsampler(subsample_step=10)],
    sdp_weight=1.0,  # Weight for descriptor preservation
    lb_weight=1e-2,  # Weight for LB commutativity
    mult_weight=1e-1,  # Weight for multiplication commutativity
    orient_weight=0.0,  # Weight for orientation commutativity
    refiners=[  # Custom refinement pipeline
        IcpRefiner(nit=5),
        ZoomOut(nit=4, step=5),
    ],
)

custom_matcher = FunctionalMapMatcher(config=config)
result = custom_matcher(mesh_a, mesh_b)

result.p2p.shape

(5000,)

## Custom refiners

You can specify any sequence of [refiners](./15_refine_functional_map.ipynb) in the config.

In [11]:
from geomfum.refine import OrthogonalRefiner

# Use orthogonal projection followed by ICP
config = MatcherConfig(
    refiners=[
        OrthogonalRefiner(),
        IcpRefiner(nit=10),
    ]
)

matcher = FunctionalMapMatcher(config=config)
result = matcher(mesh_a, mesh_b)

result.p2p.shape

(5000,)

In [12]:
# Disable refinement entirely
config = MatcherConfig(refiners=[])

matcher = FunctionalMapMatcher(config=config)
result = matcher(mesh_a, mesh_b)

result.refined_fmap  # Should be None

## Custom FeatureMatcher

You can also customize the `FeatureMatcher` with custom descriptors.

In [13]:
config = FeatureMatcherConfig(
    spectrum_size=100,
    descriptors=[
        HeatKernelSignature.from_registry(n_domain=50),
        WaveKernelSignature.from_registry(n_domain=100),
    ],
)

matcher = FeatureMatcher(config=config)
result = matcher(mesh_a, mesh_b)

result.p2p.shape

(5000,)

## Further reading

* [How to compute a functional map?](./07_functional_map.ipynb)

* [How to refine a functional map?](./15_refine_functional_map.ipynb)

* [How to create a descriptor pipeline?](./04_descriptor_pipeline.ipynb)

* [How to set landmarks?](./06_landmarks.ipynb)